# Self-Play V11 (Colab)

**Anchor games** — V11 is mostly crisis-driven; this notebook contributes ~500 long-trajectory games to keep late-game distribution covered.

Model: pillar2x2_epoch_10 (V10-trained policy_only model, +19% over 2W2).
Evaluator: feature_value_weights_2x.npz (V10 fit, R²=0.21, +29% MCTS median over V8 fit).

**Critical:** BATCH_SIZE=8 (HISTORY lesson 115).

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `pillar2x2_epoch_10.pt` — V10-era model
3. `feature_value_weights_2x.npz` — V10-fit evaluator

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar2x2_epoch_10.pt /content/alphatrain/data/model.pt
!cp {DRIVE}/feature_value_weights_2x.npz /content/alphatrain/data/feature_value_weights_2x.npz
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')
print(f'Feature weights: {os.path.getsize("/content/alphatrain/data/feature_value_weights_2x.npz")} bytes')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === SELF-PLAY V11 ANCHOR GAMES ===
# Distribute by editing SEED_START / SEED_END.
# Instance 1: 900000-900250
# Instance 2: 900250-900500
SEED_START = 900000
SEED_END = 900250
# =====================

SIMS = 600                 # V11 sim count (HISTORY lesson 124-126)
WORKERS = 24
BATCH_SIZE = 8             # MCTS quality requires bs=8 (HISTORY 115)
MAX_TURNS = 5000
SAVE_DIR = f'{DRIVE}/selfplay_v11_s600'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, Cap: {MAX_TURNS} turns, Workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/model.pt \
    --feature-value-weights /content/alphatrain/data/feature_value_weights_2x.npz \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves 5 \
    --max-turns {MAX_TURNS} \
    --compile